# L5c: Introduction to Linear Programming

How should we allocate limited resources among competing uses? We might decide how much of each product to purchase with a fixed budget, or how to send a required amount of flow through a network at minimum cost. Linear programming represents these decisions using a linear objective function, linear constraints, and bounds on continuous decision variables. The constraints describe which decisions are feasible; the objective allows us to compare them.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Formulate resource-allocation models:__ Identify the decision variables, objective coefficients, constraints, and bounds in consumer-choice and minimum-cost-flow problems. Explain how each part of the linear program represents the original allocation question.
> * __Explain duality and resource values:__ Construct the dual of a resource-allocation model and explain how feasible dual solutions bound the primal objective. Use the consumer's budget constraint to interpret the optimal dual variable as the utility gained from an additional unit of budget.
> * __Evaluate optimization results:__ Distinguish a feasible decision from an optimal one. Check solver status, recompute the objective, and verify the constraints and bounds; explain how agreement between feasible primal and dual objective values certifies optimality.

We begin with the consumer's allocation problem, work through the choice between apples and oranges, and formulate network flow using an incidence matrix. We then return to the consumer model to develop its dual and interpret the value of the budget. These examples connect the mathematical formulation with the checks needed to interpret a solver's result.

Let's get started!

___


## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package and the packages used for the allocation example.

Let's set up our code environment:


In [1]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"))


See the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5800 course package documentation](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/) for the functions and types used in the companion example.

___

## Examples

We will use the following example to connect budget allocation with the geometry of a linear program:

> [▶ Apples, Oranges, and Linear Allocation](CHEME-5800-L5c-Example-FruitAllocation-Fall-2026.ipynb). How does utility per dollar determine which fruit we should purchase? We hold prices and the budget fixed and solve three cases: apples offer more utility per dollar, oranges offer more, and both offer the same. We compare the quantities purchased, expenditure, and attained utility, then check the budget constraint and explain why equal ratios allow multiple optimal allocations.

___


## Primal Linear Programming Problems

Suppose we must choose how much of several activities to carry out while respecting limits on the resources they consume. In a __primal__ model, the decision variables describe the activities themselves: quantities purchased, production levels, or flows sent along network edges. We specify how these decisions contribute to the objective and how they use the available resources.

Let $n$ be the number of activities and $m$ the number of resource constraints. The vector $\mathbf{x}\in\mathbb{R}^{n}$ contains the activity levels, and the entry $c_i$ of the objective-coefficient vector $\mathbf{c}\in\mathbb{R}^{n}$ is the contribution per unit of activity $i$. The constraint matrix $\mathbf{A}\in\mathbb{R}^{m\times n}$ has one row per resource and one column per activity: $A_{j,i}$ is the amount of resource $j$ used per unit of activity $i$. The entry $b_j$ of the vector $\mathbf{b}\in\mathbb{R}^{m}$ gives the available amount of that resource.

> __Primal resource-allocation model:__
>
> When the activity levels are nonnegative and we want to maximize their total contribution, the linear program is given by:
> $$
> \begin{aligned}
> \underset{\mathbf{x}}{\text{maximize}}\quad
>     & O(\mathbf{x})=\mathbf{c}^{\top}\mathbf{x}
>       =\sum_{i=1}^{n}c_i x_i \\
> \text{subject to}\quad
>     & \mathbf{A}\mathbf{x}\leq\mathbf{b},\\
>     & \mathbf{x}\geq\mathbf{0}.
> \end{aligned}
> $$
> The vector inequalities apply entry by entry. Each resource constraint limits the combined use of that resource across all activities, while the nonnegativity bounds exclude negative activity levels.

For example, row $j$ of the matrix constraint expresses the following resource balance:
$$
\sum_{i=1}^{n}A_{j,i}x_i\leq b_j,
\qquad j=1,\ldots,m.
$$
Each term on the left has units of resource $j$, matching the units of $b_j$. The model is linear because each coefficient is fixed and the contributions from the activities are added together.

__What does it mean to solve this model?__ A decision vector is __feasible__ if it satisfies every constraint and bound. The collection of all feasible vectors is the __feasible region__. An optimal vector $\mathbf{x}^{\star}$ is feasible and attains the largest objective value over that region. A feasible vector need not be optimal, and several different vectors can attain the same optimal value.

If no feasible vector exists, the model is __infeasible__. If feasible vectors can produce arbitrarily large objective values, this maximization problem is __unbounded__ and has no finite optimum. These outcomes depend on the objective, constraints, and bounds together.

Linear programs can also minimize an objective, impose equality constraints, or use other lower and upper bounds. We will use the maximization form for consumer choice and an equality-constrained minimization form for network flow.

### Consumer Choice Problems as Linear Programs

Suppose we have a budget to spend on $n$ products and want to obtain the greatest total utility, where utility measures the satisfaction associated with consumption. Let $x_i\geq0$ be the quantity of product $i$ purchased, let $p_i>0$ be its price in dollars per unit, and let $u_i\geq0$ be its utility per unit. The available budget is $I\geq0$ dollars.

We treat quantities as continuous, so fractional purchases are allowed. We also assume that each additional unit contributes the same utility and that product availability does not impose another limit. Under these assumptions, the consumer-choice linear program is given by:
$$
\begin{aligned}
\underset{\mathbf{x}}{\text{maximize}}\quad
    & U(\mathbf{x})=\sum_{i=1}^{n}u_i x_i \\
\text{subject to}\quad
    & \sum_{i=1}^{n}p_i x_i\leq I,\\
    & x_i\geq0,\qquad i=1,\ldots,n.
\end{aligned}
$$
This is the resource-allocation model with one resource: the budget. The objective coefficients are the utilities, $\mathbf{c}=\mathbf{u}$, and the single row of the constraint matrix is $\mathbf{A}=[p_1\ \cdots\ p_n]$, with right-hand side $\mathbf{b}=[I]$.

### Worked Allocation: Apples and Oranges

Let's use the first case from the [allocation example](CHEME-5800-L5c-Example-FruitAllocation-Fall-2026.ipynb). Apples cost 2 dollars per unit and contribute 0.55 utility units per unit; oranges cost 4 dollars per unit and contribute 0.45 utility units per unit. With a budget of 100 dollars, we obtain the following model:
$$
\begin{aligned}
\underset{x_A,x_O}{\text{maximize}}\quad
    & U(x_A,x_O)=0.55x_A+0.45x_O \\
\text{subject to}\quad
    & 2x_A+4x_O\leq100,\\
    & x_A,x_O\geq0,
\end{aligned}
$$
where the subscripts $A$ and $O$ denote apples and oranges. We can buy at most 50 units of apples or 25 units of oranges. Mixtures are also feasible as long as their total expenditure does not exceed the budget.

__Why would an optimal allocation use the entire budget?__ Both products have positive utility, quantities are divisible, and there are no stock limits. Any unspent budget could therefore purchase more fruit and increase total utility. At an optimum, the budget constraint must hold with equality. Solving that equality for the orange quantity gives:
$$
x_O=25-\frac{1}{2}x_A,
\qquad 0\leq x_A\leq50.
$$
Substituting this expression into the objective allows us to compare all allocations that use the full budget:
$$
\begin{aligned}
U(x_A)
&=0.55x_A+0.45\left(25-\frac{1}{2}x_A\right)\\
&=11.25+0.325x_A.
\end{aligned}
$$
The coefficient of $x_A$ is positive, so utility increases as we replace oranges with apples along the budget boundary. The largest feasible apple quantity is $x_A^{\star}=50$, which gives $x_O^{\star}=0$ and $U^{\star}=27.5$ utility units. Buying only oranges is also feasible and uses the full budget, but gives only 11.25 utility units. Using all available resources does not, by itself, establish optimality.

The same preference follows by comparing __utility per dollar__, $u_i/p_i$. For these products, the ratios are given by:
$$
\frac{u_A}{p_A}=\frac{0.55}{2}=0.275,
\qquad
\frac{u_O}{p_O}=\frac{0.45}{4}=0.1125.
$$
Each dollar spent on apples contributes more utility than a dollar spent on oranges. This comparison explains the optimal allocation for the present one-budget model and will help us interpret the dual variable later. The numerical result applies to these prices, utilities, and budget; changing them can change the preferred allocation.
### Comparing Allocations and Their Geometry

What changes if the consumer values the two fruits differently? We keep the prices at $p_A=2$ and $p_O=4$ dollars per unit and the budget at $I=100$ dollars, then change the utility coefficients. The following table compares the three cases in the [allocation example](CHEME-5800-L5c-Example-FruitAllocation-Fall-2026.ipynb). The ratios measure utility per dollar, and the final column reports the maximum total utility:

<table style="margin: 12px auto; border-collapse: collapse; font-size: 95%;">
<thead>
<tr style="border-bottom: 1px solid #b8b8b8;">
<th style="padding: 5px 10px; text-align: left;">Case</th>
<th style="padding: 5px 10px; text-align: left;">Utilities<br>(<i>u</i><sub>A</sub>, <i>u</i><sub>O</sub>)</th>
<th style="padding: 5px 10px; text-align: right;">Apples<br><i>u</i><sub>A</sub>/<i>p</i><sub>A</sub></th>
<th style="padding: 5px 10px; text-align: right;">Oranges<br><i>u</i><sub>O</sub>/<i>p</i><sub>O</sub></th>
<th style="padding: 5px 10px; text-align: left;">Optimal allocation<br>(<i>x</i><sub>A</sub>, <i>x</i><sub>O</sub>)</th>
<th style="padding: 5px 10px; text-align: right;">Maximum<br>utility</th>
</tr>
</thead>
<tbody>
<tr><td style="padding: 5px 10px; text-align: left;">A</td><td style="padding: 5px 10px; text-align: left; white-space: nowrap;">(0.55, 0.45)</td><td style="padding: 5px 10px; text-align: right;">0.275</td><td style="padding: 5px 10px; text-align: right;">0.1125</td><td style="padding: 5px 10px; text-align: left;">(50, 0)</td><td style="padding: 5px 10px; text-align: right;">27.5</td></tr>
<tr><td style="padding: 5px 10px; text-align: left;">B</td><td style="padding: 5px 10px; text-align: left; white-space: nowrap;">(0.15, 0.55)</td><td style="padding: 5px 10px; text-align: right;">0.075</td><td style="padding: 5px 10px; text-align: right;">0.1375</td><td style="padding: 5px 10px; text-align: left;">(0, 25)</td><td style="padding: 5px 10px; text-align: right;">13.75</td></tr>
<tr><td style="padding: 5px 10px; text-align: left;">C</td><td style="padding: 5px 10px; text-align: left; white-space: nowrap;">(2, 4)</td><td style="padding: 5px 10px; text-align: right;">1</td><td style="padding: 5px 10px; text-align: right;">1</td><td style="padding: 5px 10px; text-align: left;">Any full-budget mixture</td><td style="padding: 5px 10px; text-align: right;">100</td></tr>
</tbody>
</table>

In Cases A and B, moving a dollar from the fruit with lower utility per dollar to the other fruit increases total utility. The optimal allocation therefore spends the entire budget on the fruit with the larger ratio. In Case C, each dollar contributes one unit of utility regardless of which fruit we purchase. Every nonnegative allocation on the budget boundary $2x_A+4x_O=100$ is optimal: buying 50 apples, buying 25 oranges, or buying 20 apples and 15 oranges gives the same total of 100 utility units. The optimal value is unique even though the optimal allocation is not. These numerical results apply to the stated prices, budget, and utility coefficients.

We can also explain these outcomes using the geometry of the feasible region. Place apples on the horizontal axis and oranges on the vertical axis. Let $m_I$ denote the slope of the budget boundary, $m_O$ the slope of a line of constant objective value, and $\bar U$ a chosen total utility. With positive prices and utilities, solving the budget and utility equations for the orange quantity gives:
$$
\begin{aligned}
\text{Budget boundary:}\qquad
x_O &= \frac{I}{p_O}
       +\underbrace{\left(-\frac{p_A}{p_O}\right)}_{m_I}x_A,\\[6pt]
\text{Constant utility:}\qquad
x_O &= \frac{\bar U}{u_O}
       +\underbrace{\left(-\frac{u_A}{u_O}\right)}_{m_O}x_A.
\end{aligned}
$$
Increasing $\bar U$ shifts the constant-utility line upward without changing its slope. We seek the largest value of $\bar U$ whose line still intersects the feasible region. The following schematic shows where this last contact occurs for the three slope comparisons:

<div>
    <center>
        <img src="figs/Fig-ThreeCases-LP-Schematic.svg" width="1000" alt="Three schematic allocation cases: an optimal apple corner, an optimal orange corner, and an entire optimal budget edge when the utility-per-dollar ratios are equal."/>
    </center>
</div>

The gray triangle is the feasible region, the black diagonal is the budget boundary, and the colored diagonal lines represent different total utility values. The yellow markings identify the optimal corner or edge. The dashed outer box represents illustrative stock limits beyond the affordable quantities, so it does not restrict the shaded region. The figure shows the relative slopes schematically; the table supplies the numerical results.

* __Case A: $|m_O|>|m_I|$.__ The constant-utility lines are steeper than the budget boundary. Their last feasible contact is the apple corner, consistent with apples having the greater utility per dollar.
* __Case B: $|m_O|<|m_I|$.__ The constant-utility lines are flatter than the budget boundary. Their last feasible contact is the orange corner, consistent with oranges having the greater utility per dollar.
* __Case C: $|m_O|=|m_I|$.__ The lines are parallel to the budget boundary. At the largest feasible utility, one of these lines coincides with the entire budget edge, so both corners and every mixture between them are optimal.

This distinction matters when we interpret a solver's answer. For Case C, different solvers can return different quantities while agreeing on the maximum utility and satisfying the same budget constraint.
### Minimum-Cost Network Flow Problems as Linear Programs

In the [maximum-flow lecture](../L5a/CHEME-5800-L5a-Lecture-MaximumFlowProblems-Fall-2026.ipynb), we asked how much flow a network could carry. Suppose we now attach a cost to sending flow along each edge. Several feasible routings may deliver the same amount, but their costs can differ. The __minimum-cost-flow problem__ chooses the least expensive routing that delivers a specified amount of flow while respecting capacities and conservation.

Let $\mathcal{G}=(\mathcal{V},\mathcal{E})$ be a finite directed graph, with vertex set $\mathcal{V}$ and edge set $\mathcal{E}$. The source $s$ supplies flow and the distinct sink $t$ receives it. We consider edges joining distinct vertices and fix an ordering of the vertices and edges. Each edge $j$ has a finite capacity $c_j\geq0$, a cost $w_j$ per unit of flow, and a decision variable $f_j$ giving its flow. The required net flow from source to sink is the specified quantity $F\geq0$; flows, capacities, and $F$ are measured in the same units.

__How do we write conservation as a matrix equation?__ The __node–edge incidence matrix__ $\mathbf{A}\in\mathbb{R}^{|\mathcal{V}|\times|\mathcal{E}|}$ has one row per vertex and one column per edge. We use the convention that incoming flow contributes positively and outgoing flow contributes negatively. For the vertex indexed by $i$ and the edge indexed by $j$, its entries are defined by:
$$
A_{i,j}=
\begin{cases}
 1, & \text{if edge }j\text{ enters vertex }i,\\
-1, & \text{if edge }j\text{ leaves vertex }i,\\
 0, & \text{otherwise}.
\end{cases}
$$
The vector $\mathbf{f}\in\mathbb{R}^{|\mathcal{E}|}$ contains the edge flows. Each entry of $\mathbf{A}\mathbf{f}$ is therefore the total inflow minus the total outflow at one vertex. At an intermediate vertex, this difference must be zero. At the source it must be $-F$, and at the sink it must be $F$. We collect these required net inflows in the vector $\mathbf{b}\in\mathbb{R}^{|\mathcal{V}|}$, defined by:
$$
b_i=
\begin{cases}
-F, & \text{at the source},\\
 F, & \text{at the sink},\\
 0, & \text{at every intermediate vertex}.
\end{cases}
$$
The negative source entry represents a net supply to the network; the positive sink entry represents receipt of that flow. Thus, $\mathbf{A}\mathbf{f}=\mathbf{b}$ combines all vertex balances in one equation.

> __Minimum-cost flow for a specified delivery:__
>
> With the required flow $F$ encoded in $\mathbf{b}$, the linear program is given by:
> $$
> \begin{aligned}
> \underset{\mathbf{f}}{\text{minimize}}\quad
>     & \sum_{j\in\mathcal{E}}w_j f_j \\
> \text{subject to}\quad
>     & \mathbf{A}\mathbf{f}=\mathbf{b},\\
>     & 0\leq f_j\leq c_j,\qquad j\in\mathcal{E}.
> \end{aligned}
> $$
> The objective adds the costs incurred on all edges. The equalities enforce the required delivery and conservation, while the bounds keep every edge flow within its capacity.

This formulation applies to general directed networks. In an assignment application, the edges and their capacities represent which worker–task combinations are available; in a transport application, they represent the available routes.

### Worked Network: Choosing Between Two Routes

Consider the three-vertex network below. Edges 1 and 2 form a route from $s$ through $v$ to $t$, while edge 3 goes directly from $s$ to $t$. Each edge label gives its capacity in flow units and its cost in dollars per flow unit:

<div>
    <center>
        <img src="figs/Fig-MinCostFlow-ThreeNode.svg" width="650" alt="Three-vertex flow network: edge 1 goes from s to v with capacity 2 and cost 1, edge 2 goes from v to t with capacity 2 and cost 1, and edge 3 goes directly from s to t with capacity 3 and cost 5."/>
    </center>
</div>

The route through $v$ costs 2 dollars per unit delivered because each unit traverses two edges costing 1 dollar each. The direct route costs 5 dollars per unit. The cheaper route can carry only 2 units, so some of a 3-unit delivery must travel directly.

Order the rows as $(s,v,t)$ and the columns as edges $(1,2,3)$. For a required flow $F=3$, the conservation equation is given by:
$$
\underbrace{
\begin{bmatrix}
-1&0&-1\\
 1&-1&0\\
 0&1&1
\end{bmatrix}}_{\mathbf{A}}
\underbrace{
\begin{bmatrix}f_1\\f_2\\f_3\end{bmatrix}}_{\mathbf{f}}
=
\underbrace{
\begin{bmatrix}-3\\0\\3\end{bmatrix}}_{\mathbf{b}}.
$$
The first row requires $f_1+f_3=3$ units to leave the source. The second row requires $f_1=f_2$ at the intermediate vertex, and the third requires $f_2+f_3=3$ units to enter the sink. Each edge column has a $-1$ where that edge begins and a $+1$ where it ends.

To calculate the least expensive routing, let $q$ be the flow sent through $v$. Conservation gives $f_1=f_2=q$ and $f_3=3-q$. The edge capacities restrict $q$ to $0\leq q\leq2$. Substituting into the total cost gives:
$$
C(q)=q+q+5(3-q)=15-3q.
$$
The cost decreases as $q$ increases, so we use the full capacity of the cheaper route. The optimal flows are $(f_1,f_2,f_3)=(2,2,1)$, with a total cost of 9 dollars. Sending all 3 units directly would also be feasible, but would cost 15 dollars. Conservation and capacity checks establish that a routing is feasible; the cost comparison determines which feasible routing is preferred.

__How do we obtain a minimum-cost maximum flow?__ First, let $F_{\max}$ denote the largest feasible net flow from $s$ to $t$. We can solve the combined problem in two steps:

1. Solve the maximum-flow problem to determine $F_{\max}$.
2. Set $F=F_{\max}$ in the minimum-cost model and minimize cost among routings that deliver this amount.

In this network, the two edges leaving the source have capacities of 2 and 3 units, giving an upper bound of 5 units. The feasible flow $(2,2,3)$ attains that bound, so $F_{\max}=5$. Delivering this maximum amount costs 19 dollars. The earlier 9-dollar solution delivers the specified 3 units. A requirement larger than $F_{\max}$ would make the model infeasible.

The [L5d lab](../L5d/CHEME-5800-L5d-Lab-MinimumCostAssignmentFlow-Fall-2026.ipynb) uses this incidence convention to formulate a worker–task assignment model, minimize the cost of a required flow, and examine what changes when an assignment becomes unavailable.

___


## Dual Linear Programming Problems

How can we show that no feasible decision is better than our candidate? A __dual linear program__ constructs a bound on what the primal can achieve by assigning values to its resources.

### What Is the Consumer's Budget Worth?

Let $y\geq0$ be the value of one dollar of budget, measured in utility units per dollar. For a product with price $p_i$ and utility $u_i$ per unit, require $p_i y\geq u_i$ so the assigned value of its cost covers its utility. Since total expenditure cannot exceed the budget $I$, total utility cannot exceed $Iy$. The __consumer dual__ finds the smallest such bound:
$$
\begin{aligned}
\underset{y}{\text{minimize}}\quad
    & Iy \\
\text{subject to}\quad
    & p_i y\geq u_i,\qquad i=1,\ldots,n,\\
    & y\geq0.
\end{aligned}
$$
For our apples-and-oranges case, the constraints are $2y\geq0.55$ and $4y\geq0.45$. With $I=100$, the optimal budget value and utility bound are given by:
$$
y^{\star}=\max\!\left\{\frac{0.55}{2},\frac{0.45}{4}\right\}=0.275,
\qquad
Iy^{\star}=27.5.
$$
Buying 50 apples and no oranges attains 27.5 utility units. Matching the dual bound proves that this purchase is optimal.

The budget's __shadow price__ is $y^{\star}=0.275$ utility units per dollar: an additional dollar buys 0.5 units of apples and increases optimal utility by 0.275. This interpretation assumes fixed prices and utilities, divisible quantities, and sufficient product availability. In more general allocation models, shadow prices can change as resource availability changes.

### The General Primal–Dual Pair

Return to the resource-allocation model with $n$ activities, $m$ resource constraints, and $\mathbf{A}\in\mathbb{R}^{m\times n}$. The entry $A_{j,i}$ describes the use of resource $j$ by one unit of activity $i$, $b_j$ is the available amount, and $c_i$ is the activity's contribution to the objective. Let $y_j\geq0$ be an assigned value per unit of resource $j$, in objective units per resource unit, and collect these values in $\mathbf{y}\in\mathbb{R}^{m}$.

> __Resource-allocation primal and dual:__
>
> For a maximization model with nonnegative activities and upper resource limits, the paired linear programs are given by:
> $$
> \begin{aligned}
> \text{Primal:}\quad
> &\underset{\mathbf{x}}{\text{maximize}}\quad\mathbf{c}^{\top}\mathbf{x}
> &\qquad\text{Dual:}\quad
> &\underset{\mathbf{y}}{\text{minimize}}\quad\mathbf{b}^{\top}\mathbf{y}\\
> &\text{subject to}\quad\mathbf{A}\mathbf{x}\leq\mathbf{b}
> &&\text{subject to}\quad\mathbf{A}^{\top}\mathbf{y}\geq\mathbf{c},\\
> &\hphantom{\text{subject to}\quad}\mathbf{x}\geq\mathbf{0}
> &&\hphantom{\text{subject to}\quad}\mathbf{y}\geq\mathbf{0}.
> \end{aligned}
> $$
> There is one dual variable for each primal resource constraint and one dual constraint for each primal activity. The dual minimizes the total assigned value of the available resources.

For activity $i$, the dual constraint is $\sum_{j=1}^{m}A_{j,i}y_j\geq c_i$. It requires the assigned value of all resources used by that activity to cover its objective contribution. The transpose appears because we now combine entries down each activity column of $\mathbf{A}$, rather than across each resource row.

The argument used for the consumer model also applies here. For any primal-feasible $\mathbf{x}$ and dual-feasible $\mathbf{y}$, we obtain the following bound:
$$
\mathbf{c}^{\top}\mathbf{x}
\leq(\mathbf{A}^{\top}\mathbf{y})^{\top}\mathbf{x}
=\mathbf{y}^{\top}\mathbf{A}\mathbf{x}
\leq\mathbf{y}^{\top}\mathbf{b}.
$$
This is __weak duality__. The first inequality uses dual feasibility and $\mathbf{x}\geq\mathbf{0}$; the last uses primal feasibility and $\mathbf{y}\geq\mathbf{0}$. Every feasible dual objective value is therefore an upper bound on every feasible primal objective value.

The __duality gap__ between these feasible candidates is $\mathbf{b}^{\top}\mathbf{y}-\mathbf{c}^{\top}\mathbf{x}\geq0$. A zero gap proves that both candidates are optimal, just as matching the attained utility to the dual bound did in the consumer example.

> __Strong duality for linear programs:__
>
> If the primal is feasible and has a finite optimal value, the dual also has an optimal solution. Their optimal objective values satisfy:
> $$
> \mathbf{c}^{\top}\mathbf{x}^{\star}
> =\mathbf{b}^{\top}\mathbf{y}^{\star}.
> $$
> Here, $\mathbf{x}^{\star}$ and $\mathbf{y}^{\star}$ denote optimal primal and dual solutions. The equality concerns their objective values; the two vectors describe different quantities and can have different dimensions.

Weak duality establishes the bound used to check a candidate solution. Strong duality states that, when a finite optimum exists, optimal solutions attain that bound. We use the result here; a proof of strong duality is beyond the present development.

### Constraint Directions and Variable Signs

The paired models above use a particular maximization convention. Their signs follow from the inequalities in the bound, so they must be reconsidered when a model uses other constraint or variable types:

* A primal resource constraint of the form $\sum_i A_{j,i}x_i\leq b_j$ gives a nonnegative dual variable $y_j\geq0$. Multiplying the constraint by $y_j$ then preserves its inequality direction.
* An equality constraint $\sum_i A_{j,i}x_i=b_j$ gives a __free__ dual variable $y_j$, which may be positive, negative, or zero. Multiplication by a value of either sign preserves the equality.
* A nonnegative primal variable $x_i\geq0$ gives the dual constraint $(\mathbf{A}^{\top}\mathbf{y})_i\geq c_i$.
* A free primal variable gives the equality $(\mathbf{A}^{\top}\mathbf{y})_i=c_i$. Equal coefficients are needed for the bound to hold when $x_i$ can have either sign.

The equality balances in the network-flow model illustrate why dual variables need not always be nonnegative. The constraint type and variable bounds determine the appropriate dual form; they cannot be replaced by a blanket instruction to reverse every inequality.

___



## How a Solver Fits the Modeling Workflow

We supply the solver with decision variables, an objective, constraints, and bounds. It reports why the calculation stopped and, when available, returns a candidate solution. Before interpreting that solution, we make the following checks:

1. __Read the status.__ An optimal status reports optimality within the solver's numerical tolerances. A time or iteration limit may leave a feasible candidate without establishing optimality. Confirm that a feasible solution is available before interpreting variable values; an infeasible or unbounded model has no finite optimal allocation. The [JuMP solution guide](https://jump.dev/JuMP.jl/stable/manual/solutions/) explains the distinction between termination and solution status.
2. __Verify feasibility.__ Substitute the candidate into the original constraints and bounds. For the consumer model, check nonnegative quantities and expenditure against the budget. For network flow, check every node balance in $\mathbf{A}\mathbf{f}=\mathbf{b}$ and every edge bound $0\leq f_j\leq c_j$.
3. __Recompute the objective.__ Calculate utility or flow cost from the returned quantities and the original coefficients, then compare it with the reported objective. This checks that we are interpreting the variables and their ordering correctly.
4. __Check an optimality bound.__ For our resource-allocation maximization model, a feasible dual candidate gives the upper bound $\mathbf{b}^{\top}\mathbf{y}$. Verify dual feasibility before comparing it with $\mathbf{c}^{\top}\mathbf{x}$. A zero gap certifies optimality in exact arithmetic; numerical comparisons require stated tolerances.

Choose tolerances that account for the units and scales of the quantities being compared. The [fruit-allocation example](CHEME-5800-L5c-Example-FruitAllocation-Fall-2026.ipynb) applies these checks using the analytical utility bound derived from utility per dollar.

How does an algorithm find a candidate? The supporting [▶ Revised Simplex notebook](CHEME-5800-L5c-Algorithm-RevisedSimplex-Fall-2026.ipynb) explains how a feasible basis represents a corner, reduced costs identify a potentially improving direction, and the ratio test limits the step to preserve feasibility. It provides a closer look at solver mechanics; implementing a production solver is beyond this week's scope.

___



## Summary

In this lecture, we used consumer-choice and network-flow problems to develop linear programming models, then introduced duality to bound the objective and interpret the value of a limited resource.

> __Key Takeaways:__
>
> * __Formulating allocation models:__ We expressed consumer purchases and network flows using continuous decision variables, linear objectives, and resource or balance constraints. The flow example distinguished minimizing the cost of a required delivery from first finding the maximum possible flow.
> * __Interpreting duality and resource values:__ We used feasible dual solutions to bound the primal objective and interpreted optimal dual variables as shadow prices. This connected the allocation decision with the marginal value of additional resources.
> * __Evaluating optimization results:__ We distinguished feasibility from optimality and identified the checks needed to interpret solver output: status, constraints, bounds, and a recomputed objective. Agreement between feasible primal and dual objective values supplied an optimality certificate.

The primal asks how best to use the available resources, and the dual assigns values that bound what those resources can achieve. When a feasible allocation reaches that bound, we have justified its optimality.
